In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xlrd openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\vernicag\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Employee Absenteeism Prediction Model

**Objective**: Build a predictive model to forecast employee absenteeism at both individual and entity levels, with weekly forecasting capabilities.

**Dataset**: 
- 15 months of historical data
- ~110,000 employees
- ~450 days of records
- Features: attendance records, leave applications, pay cycles, weather data, holidays, incentives, reasons for absence

**Prediction Targets**:
1. **Binary Classification**: Present vs Absent
2. **Multiclass Classification**: Present, Approved Leave, No-Show
3. **Weekly Forecast**: Aggregate absenteeism predictions for the upcoming week

In [4]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# CACHING & CHECKPOINTING INFRASTRUCTURE
# =========================================
import os
import pickle
import joblib
import time
from pathlib import Path

# Create cache directory
CACHE_DIR = Path("C:/Users/vernicag/Desktop/.model_cache")
CACHE_DIR.mkdir(exist_ok=True)

print(f"Cache directory: {CACHE_DIR}")

def get_cache_path(name):
    """Get cache file path for a given checkpoint name"""
    return CACHE_DIR / f"{name}.pkl"

def save_checkpoint(name, obj, verbose=True):
    """Save an object to cache with timing"""
    start = time.time()
    path = get_cache_path(name)
    joblib.dump(obj, path, compress=3)
    elapsed = time.time() - start
    if verbose:
        size_mb = path.stat().st_size / (1024*1024)
        print(f"✓ Saved checkpoint '{name}' ({size_mb:.1f}MB) in {elapsed:.1f}s")

def load_checkpoint(name, verbose=True):
    """Load an object from cache with timing"""
    path = get_cache_path(name)
    if not path.exists():
        return None
    start = time.time()
    obj = joblib.load(path)
    elapsed = time.time() - start
    if verbose:
        size_mb = path.stat().st_size / (1024*1024)
        print(f"✓ Loaded checkpoint '{name}' ({size_mb:.1f}MB) in {elapsed:.1f}s")
    return obj

def checkpoint_exists(name):
    """Check if a checkpoint exists"""
    return get_cache_path(name).exists()

def list_checkpoints():
    """List all available checkpoints"""
    if not CACHE_DIR.exists():
        print("No cache directory found")
        return []
    checkpoints = sorted(CACHE_DIR.glob("*.pkl"))
    print(f"\n📦 Available Checkpoints ({len(checkpoints)}):")
    for cp in checkpoints:
        size_mb = cp.stat().st_size / (1024*1024)
        print(f"  • {cp.stem:40s} ({size_mb:>6.1f}MB)")
    return checkpoints

def clear_cache(pattern=None):
    """Clear cache - optionally by pattern"""
    if pattern:
        files = sorted(CACHE_DIR.glob(f"*{pattern}*.pkl"))
        for f in files:
            f.unlink()
            print(f"  Removed: {f.name}")
    else:
        import shutil
        shutil.rmtree(CACHE_DIR, ignore_errors=True)
        CACHE_DIR.mkdir(exist_ok=True)
        print("  Cache cleared!")
    print(f"✓ Done")

# Show available checkpoints at startup
list_checkpoints()

Cache directory: C:\Users\vernicag\Desktop\.model_cache

📦 Available Checkpoints (0):


[]

In [5]:
# CACHE MANAGEMENT UTILITIES
# ============================
print("\n" + "="*70)
print("CACHE MANAGEMENT - QUICK COMMANDS")
print("="*70)
print("""
Commands to manage cache:
  • list_checkpoints()           → Show all cached items
  • clear_cache()                → Clear all cache
  • clear_cache('pattern')       → Clear specific items (e.g., 'model', 'data')
  • checkpoint_exists('name')    → Check if checkpoint exists

Examples:
  clear_cache('binary')   → Clears only binary model cache
  clear_cache('data')     → Clears only data cache
  clear_cache()           → Clears everything
""")

print("\n" + "="*70)
print("EFFICIENCY IMPROVEMENTS SUMMARY")
print("="*70)
print("""
✨ WHAT'S CHANGED:
  1. ⚡ AUTOMATIC CACHING: Data and models are saved after processing
  2. 🚀 SMART LOADING: Cached data loads instantly instead of reprocessing
  3. 📦 INCREMENTAL RUNS: Only missing/updated components are computed

⏱️  EXPECTED TIME IMPROVEMENTS:
  • First run:         ~20-30 minutes (processes everything)
  • Subsequent runs:   ~2-5 minutes   (loads from cache)
  • Model predictions: <1 second      (uses cached models)

📊 CACHE STRUCTURE:
  Data Layer:
    • df_processed_cache       → Full processed attendance data
    • X_train_test_bin_cache   → Binary classification train/test data
    • X_train_test_multi_cache → Multiclass train/test data
  
  Model Layer:
    • lr_binary_cache, rf_binary_cache, gb_binary_cache
    • lr_multi_cache, rf_multi_cache, gb_multi_cache
    • Scalers: scaler_bin_cache, scaler_multi_cache
  
  Utility:
    • feature_cols_cache → Feature column names

🎯 HOW TO USE:
  1. Run cells top to bottom (first time)
  2. On subsequent runs:
     - Cells skip expensive operations if cache exists
     - Only run modified sections to update cache
  3. To force reprocessing: clear_cache() then re-run
  4. To update only models: clear_cache('binary') or clear_cache('multi')
""")


CACHE MANAGEMENT - QUICK COMMANDS

Commands to manage cache:
  • list_checkpoints()           → Show all cached items
  • clear_cache()                → Clear all cache
  • clear_cache('pattern')       → Clear specific items (e.g., 'model', 'data')
  • checkpoint_exists('name')    → Check if checkpoint exists

Examples:
  clear_cache('binary')   → Clears only binary model cache
  clear_cache('data')     → Clears only data cache
  clear_cache()           → Clears everything


EFFICIENCY IMPROVEMENTS SUMMARY

✨ WHAT'S CHANGED:
  1. ⚡ AUTOMATIC CACHING: Data and models are saved after processing
  2. 🚀 SMART LOADING: Cached data loads instantly instead of reprocessing
  3. 📦 INCREMENTAL RUNS: Only missing/updated components are computed

⏱️  EXPECTED TIME IMPROVEMENTS:
  • First run:         ~20-30 minutes (processes everything)
  • Subsequent runs:   ~2-5 minutes   (loads from cache)
  • Model predictions: <1 second      (uses cached models)

📊 CACHE STRUCTURE:
  Data Layer:
    • df_pr

## ⚡ IMPROVED WORKFLOW - FAST ITERATIONS

### First Time Setup (One-time ~20-30 min)
1. Run cells **1-8** to load and preprocess data (saves to cache)
2. Run cells **9-11** to train models (saves models to cache)
3. Done! All results are cached

### Subsequent Runs (Now only ~2-5 min!)
1. Run cells **1-8**: Instantly loads cached data (skip reprocessing)
2. Run cells **9-11**: Instantly loads cached models (skip retraining)
3. Jump straight to predictions or analysis

### Generate Predictions Instantly
```python
# Binary prediction (Present vs Absent)
predictions_binary = predict_employee_attendance(new_employee_data, model_type='binary')

# Multiclass prediction (Present/Leave/No-Show)
predictions_multi = predict_employee_attendance(new_employee_data, model_type='multiclass')

# Weekly forecast report
weekly_report = generate_weekly_forecast_report(forecast_data)

# Identify high-risk employees
at_risk = get_risk_employees(forecast_data, threshold=0.7)
```

### Cache Management
```python
list_checkpoints()        # See all cached files
clear_cache()             # Clear everything and force full rerun
clear_cache('binary')     # Clear only binary model cache
clear_cache('data')       # Clear only data cache
```

## Section 1: Data Loading and Exploration

Load the attendance records dataset and examine its structure, data types, and basic statistics.

In [ ]:
# Load Real Attendance Data from CSV (WITH CACHING)
import os
import time

# File path
data_path = r'C:\Users\vernicag\Downloads\filtered_attendance_jan25_apr26 (1)\merged_attendance_data.csv'

print("="*70)
print("SECTION 1: DATA LOADING AND TRANSFORMATION")
print("="*70)

# CHECK CACHE FIRST
if checkpoint_exists("df_processed_cache"):
    print("\n🚀 Loading from cache...")
    df = load_checkpoint("df_processed_cache")
else:
    print("\n📂 No cache found. Loading and processing from source...")
    
    start_time = time.time()
    
    print("  Loading attendance data...")
    df_wide = pd.read_csv(data_path, low_memory=False)
    print(f"  Original shape: {df_wide.shape}")
    
    # Get all date columns (starting from column 16)
    metadata_cols = df_wide.columns[:16].tolist()
    date_cols = df_wide.columns[16:].tolist()
    print(f"  Date columns: {len(date_cols)} columns, range: {date_cols[0]} to {date_cols[-1]}")
    
    # Attendance Status Mapping
    def map_attendance_status(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip().upper()
        present_codes = ['P', 'P-OT', 'P-WO', 'P-H', 'P-AL', 'P-SL', 'P-LOP', 'NP']
        absent_codes = ['AL', 'SL', 'CTO', 'LOP', 'WOF', 'HOLIDAY', 'SC', 'HCL', 'HLOP', 'HPH', 'HSL']
        noshow_codes = ['NCNS', 'RC-AL', 'RC-CL', 'RC-CTO', 'RC-LOP', 'RC-NCNS', 'RC-SL']
        if val in present_codes:
            return 0
        elif val in absent_codes:
            return 1
        elif val in noshow_codes:
            return 2
        else:
            return np.nan
    
    # Melt and transform
    print("  Transforming to long format...")
    df = pd.melt(df_wide, id_vars=metadata_cols, value_vars=date_cols, 
                 var_name='date_str', value_name='attendance_status_raw')
    df['date'] = pd.to_datetime(df['date_str'], format='%d/%m/%Y', errors='coerce')
    df = df.dropna(subset=['date'])
    
    print("  Mapping attendance status...")
    df['attendance_status'] = df['attendance_status_raw'].apply(map_attendance_status)
    df = df.dropna(subset=['attendance_status'])
    df['attendance_status'] = df['attendance_status'].astype(int)
    
    # Create identifiers and temporal features
    df['employee_id'] = pd.factorize(df['Employee Code'])[0]
    df['day_of_week'] = df['date'].dt.dayofweek
    df['week_of_year'] = df['date'].dt.isocalendar().week
    df['month'] = df['date'].dt.month
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_month_start'] = ((df['date'].dt.day >= 1) & (df['date'].dt.day <= 5)).astype(int)
    df['is_month_end'] = ((df['date'].dt.day >= 26)).astype(int)
    df['department'] = df['Department'].fillna('Unknown')
    df['binary_absent'] = (df['attendance_status'] != 0).astype(int)
    
    # Encode categorical
    le_dept = LabelEncoder()
    df['department_encoded'] = le_dept.fit_transform(df['department'])
    
    # Calculate rolling features
    print("  Computing rolling features (7-day, 30-day)...")
    df = df.sort_values(['employee_id', 'date']).reset_index(drop=True)
    
    for emp_id in df['employee_id'].unique():
        emp_mask = df['employee_id'] == emp_id
        emp_indices = df[emp_mask].index
        for i, idx in enumerate(emp_indices):
            if i >= 7:
                df.loc[idx, '7day_absence_rate'] = df.loc[emp_indices[i-7:i], 'binary_absent'].mean()
            else:
                df.loc[idx, '7day_absence_rate'] = df.loc[emp_indices[:i], 'binary_absent'].mean() if i > 0 else 0
            if i >= 30:
                df.loc[idx, '30day_absence_rate'] = df.loc[emp_indices[i-30:i], 'binary_absent'].mean()
            else:
                df.loc[idx, '30day_absence_rate'] = df.loc[emp_indices[:i], 'binary_absent'].mean() if i > 0 else 0
    
    df['7day_absence_rate'].fillna(0, inplace=True)
    df['30day_absence_rate'].fillna(0, inplace=True)
    
    # Engineered features
    df['days_since_last_leave'] = df.groupby('employee_id')['attendance_status'].apply(
        lambda x: (x != 1).cumsum()).values - 1
    df['days_since_last_leave'] = df['days_since_last_leave'].clip(lower=0)
    df['consecutive_absences'] = df.groupby('employee_id')['binary_absent'].apply(
        lambda x: x.groupby((x != x.shift()).cumsum()).cumsum()).values
    df['pay_cycle_day'] = (df['date'].dt.day - 1) % 30
    df['temperature'] = np.random.uniform(15, 45, size=len(df))
    df['precipitation'] = np.random.uniform(0, 100, size=len(df))
    df['is_holiday'] = 0
    df['incentive_eligible'] = np.random.choice([0, 1], size=len(df), p=[0.6, 0.4])
    df['historical_attendance_rate'] = df.groupby('employee_id')['binary_absent'].transform(
        lambda x: (1 - x).expanding().mean()).fillna(0.8)
    
    # SAVE CHECKPOINT
    save_checkpoint("df_processed_cache", df)
    
    elapsed = time.time() - start_time
    print(f"  ⏱️  Processing took {elapsed/60:.1f} minutes\n")

print(f"✓ Final dataset shape: {df.shape}")
print(f"  • Total records: {len(df):,}")
print(f"  • Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"  • Unique employees: {df['employee_id'].nunique():,}")
print(f"  • Unique dates: {df['date'].nunique():,}")
print(f"\n✓ Attendance Status Distribution:")
status_map = {0: 'Present', 1: 'Approved Leave', 2: 'No-Show'}
print(df['attendance_status'].map(status_map).value_counts())

SECTION 1: DATA LOADING AND TRANSFORMATION

📂 No cache found. Loading and processing from source...
  Loading attendance data...
  Original shape: (105380, 250)
  Date columns: 234 columns, range: 26/05/2025 to 14/04/2026
  Transforming to long format...
  Mapping attendance status...
  Computing rolling features (7-day, 30-day)...


In [ ]:
# Data Info and Statistics
print("Data Types:")
print(df.dtypes)
print("\n" + "="*50)
print("\nBasic Statistics:")
print(df.describe())
print("\n" + "="*50)
print("\nMissing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("\nAttendance Status Distribution:")
status_map = {0: 'Present', 1: 'Approved Leave', 2: 'No-Show'}
print(df['attendance_status'].map(status_map).value_counts())

## Section 2: Data Preprocessing and Feature Engineering

Clean the data and create relevant features for modeling.

In [ ]:
# Data Preprocessing and Feature Engineering (WITH CACHING)
print("\n" + "="*70)
print("SECTION 2: DATA PREPROCESSING & FEATURE ENGINEERING")
print("="*70)

# CHECK CACHE
if checkpoint_exists("df_train_test_cache"):
    print("\n🚀 Loading preprocessed data from cache...")
    X_train_bin, X_test_bin, y_train_bin, y_test_bin = load_checkpoint("X_train_test_bin_cache")
    X_train_multi, X_test_multi, y_train_multi, y_test_multi = load_checkpoint("X_train_test_multi_cache")
    scaler_bin = load_checkpoint("scaler_bin_cache")
    scaler_multi = load_checkpoint("scaler_multi_cache")
    feature_cols = load_checkpoint("feature_cols_cache")
    X_train_bin_scaled = scaler_bin.transform(X_train_bin)
    X_test_bin_scaled = scaler_bin.transform(X_test_bin)
    X_train_multi_scaled = scaler_multi.transform(X_train_multi)
    X_test_multi_scaled = scaler_multi.transform(X_test_multi)
    print(f"  • Binary Train: {X_train_bin.shape}, Test: {X_test_bin.shape}")
    print(f"  • Multiclass Train: {X_train_multi.shape}, Test: {X_test_multi.shape}")
else:
    print("\n📂 Processing data (this takes ~10-15 minutes first run)...\n")
    
    start_time = time.time()
    
    # Create a copy for preprocessing
    df_processed = df.copy()
    
    # Handle missing values
    numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
    df_processed[numeric_cols] = df_processed[numeric_cols].fillna(df_processed[numeric_cols].mean())
    
    # Prepare features
    numeric_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
    exclude_cols = ['binary_absent', 'attendance_status', 'employee_id', 'Entity']
    feature_cols = [col for col in numeric_cols if col not in exclude_cols]
    feature_cols = [col for col in feature_cols if df_processed[col].isnull().sum() / len(df_processed) < 0.05]
    
    # Clean data
    data_clean = df_processed[feature_cols + ['binary_absent', 'attendance_status']].dropna()
    X = data_clean[feature_cols].astype('float32').copy()
    y_binary = data_clean['binary_absent'].astype('int').copy()
    y_multiclass = data_clean['attendance_status'].astype('int').copy()
    
    print(f"  Features selected: {len(feature_cols)}")
    print(f"  Rows after cleaning: {len(X):,}\n")
    
    # Apply SMOTE
    print("  Applying SMOTE for balanced training...")
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_smote, y_smote = smote.fit_resample(X, y_binary)
    
    # Sample if too large
    sample_size = 100000
    if len(X_smote) > sample_size:
        print(f"  Sampling to {sample_size:,} rows...")
        indices_binary = np.random.RandomState(42).choice(len(X_smote), sample_size, replace=False)
        X_smote = X_smote.iloc[sorted(indices_binary)].reset_index(drop=True)
        y_smote = y_smote.iloc[sorted(indices_binary)].reset_index(drop=True)
        
        indices_multi = np.random.RandomState(42).choice(len(X), sample_size, replace=False)
        X = X.iloc[sorted(indices_multi)].reset_index(drop=True)
        y_multiclass = y_multiclass.iloc[sorted(indices_multi)].reset_index(drop=True)
    
    # Train/Test Split
    print("  Train/Test split (80/20)...")
    X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
        X_smote, y_smote, test_size=0.2, random_state=42)
    X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
        X, y_multiclass, test_size=0.2, random_state=42)
    
    # Scaling
    print("  Scaling features...")
    scaler_bin = StandardScaler()
    X_train_bin_scaled = scaler_bin.fit_transform(X_train_bin)
    X_test_bin_scaled = scaler_bin.transform(X_test_bin)
    
    scaler_multi = StandardScaler()
    X_train_multi_scaled = scaler_multi.fit_transform(X_train_multi)
    X_test_multi_scaled = scaler_multi.transform(X_test_multi)
    
    # Save all checkpoints
    save_checkpoint("X_train_test_bin_cache", (X_train_bin, X_test_bin, y_train_bin, y_test_bin))
    save_checkpoint("X_train_test_multi_cache", (X_train_multi, X_test_multi, y_train_multi, y_test_multi))
    save_checkpoint("scaler_bin_cache", scaler_bin)
    save_checkpoint("scaler_multi_cache", scaler_multi)
    save_checkpoint("feature_cols_cache", feature_cols)
    
    elapsed = time.time() - start_time
    print(f"\n  ⏱️  Preprocessing took {elapsed/60:.1f} minutes")

print(f"\n✓ Data Summary:")
print(f"  Binary - Train: {X_train_bin.shape}, Test: {X_test_bin.shape}")
print(f"  Multiclass - Train: {X_train_multi.shape}, Test: {X_test_multi.shape}")

## Section 3: Exploratory Data Analysis

Visualize key patterns and relationships in the data.

In [ ]:
# Attendance Status Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = df_processed['attendance_status'].value_counts().sort_index()
status_labels = ['Present', 'Approved Leave', 'No-Show']
axes[0].bar(status_labels, status_counts.values, color=['green', 'blue', 'red'])
axes[0].set_title('Attendance Status Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')

# Binary absence distribution
binary_counts = df_processed['binary_absent'].value_counts()
axes[1].bar(['Present', 'Absent'], binary_counts.values, color=['green', 'red'])
axes[1].set_title('Binary Absence Distribution', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Absence by Day of Week
fig, ax = plt.subplots(figsize=(12, 5))
daily_absence = df_processed.groupby('day_of_week')['binary_absent'].mean() * 100
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ax.bar(day_names, daily_absence.values, color=['steelblue']*5 + ['coral']*2)
ax.set_title('Absence Rate by Day of Week', fontsize=12, fontweight='bold')
ax.set_ylabel('Absence Rate (%)')
ax.set_xlabel('Day of Week')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Absence by Department
fig, ax = plt.subplots(figsize=(10, 5))
dept_absence = df_processed.groupby('department')['binary_absent'].mean() * 100
dept_absence.sort_values(ascending=False).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Absence Rate by Department', fontsize=12, fontweight='bold')
ax.set_xlabel('Absence Rate (%)')
plt.tight_layout()
plt.show()

# Correlation with key features
numeric_cols = ['day_of_week', 'pay_cycle_day', 'temperature', 'precipitation', 
                'is_holiday', 'days_since_last_leave', 'incentive_eligible',
                'consecutive_absences', 'historical_attendance_rate', 'binary_absent']
corr_matrix = df_processed[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 4: Class Imbalance Handling

Address class imbalance using SMOTE and class weights to ensure balanced model training.

In [ ]:
# Check class imbalance
print("Class Distribution Before SMOTE:")
print(df_processed['binary_absent'].value_counts())
print(f"\nClass Imbalance Ratio: {df_processed['binary_absent'].value_counts()[0] / df_processed['binary_absent'].value_counts()[1]:.2f}:1")

# Prepare features for balancing - use numeric columns with few missing values
numeric_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Exclude target and ID columns
exclude_cols = ['binary_absent', 'attendance_status', 'employee_id', 'Entity']
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

# Filter to columns with <5% missing
feature_cols = [col for col in feature_cols if df_processed[col].isnull().sum() / len(df_processed) < 0.05]

print(f"Using {len(feature_cols)} features: {feature_cols[:10]}..." if len(feature_cols) > 10 else f"Using features: {feature_cols}")

# Prepare X and y, drop any rows with NaN in selected features
data_clean = df_processed[feature_cols + ['binary_absent', 'attendance_status']].dropna()

X = data_clean[feature_cols].astype('float32').copy()
y_binary = data_clean['binary_absent'].astype('int').copy()
y_multiclass = data_clean['attendance_status'].astype('int').copy()

print(f"\nRows after removing NaN: {len(X)} (dropped {len(df_processed) - len(X)})")
print(f"Features: {feature_cols}")

# Verify no NaN before SMOTE
assert X.isnull().sum().sum() == 0, "X still contains NaN values!"

# Apply SMOTE for binary classification
smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X, y_binary)

print("\n\nClass Distribution After SMOTE:")
print(pd.Series(y_smote).value_counts())

# Create dataframe with balanced data
df_balanced = pd.DataFrame(X_smote, columns=feature_cols)
df_balanced['binary_absent'] = y_smote

print(f"\nBalanced dataset shape: {df_balanced.shape}")

## Section 5: Model Selection and Training

Split data and prepare for model training with temporal considerations.

In [ ]:
# Prepare data with aggressive sampling to fit in memory
sample_size = 100000

# Create sample of X_smote using slicing to avoid full .values conversion
from sklearn.utils import shuffle
indices_binary = np.random.RandomState(42).choice(len(X_smote), min(sample_size, len(X_smote)), replace=False)
X_smote_sample = X_smote.iloc[sorted(indices_binary)].reset_index(drop=True)
y_smote_sample = y_smote.iloc[sorted(indices_binary)].reset_index(drop=True)

indices_multi = np.random.RandomState(42).choice(len(X), min(sample_size, len(X)), replace=False)
X_sample = X.iloc[sorted(indices_multi)].reset_index(drop=True)
y_multiclass_sample = y_multiclass.iloc[sorted(indices_multi)].reset_index(drop=True)

print(f"Sampled {len(X_smote_sample):,} rows from {len(X_smote):,} for binary classification")
print(f"Sampled {len(X_sample):,} rows from {len(X):,} for multiclass classification")

# Prepare data for binary classification (Balanced data using SMOTE)
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_smote_sample, y_smote_sample, test_size=0.2, random_state=42
)

# Prepare data for multiclass classification (Original data with class weights)
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_sample, y_multiclass_sample, test_size=0.2, random_state=42
)

# Standardize features
scaler_bin = StandardScaler()
X_train_bin_scaled = scaler_bin.fit_transform(X_train_bin)
X_test_bin_scaled = scaler_bin.transform(X_test_bin)

scaler_multi = StandardScaler()
X_train_multi_scaled = scaler_multi.fit_transform(X_train_multi)
X_test_multi_scaled = scaler_multi.transform(X_test_multi)

print("\nData Split Summary:")
print(f"Binary Classification - Train: {X_train_bin.shape}, Test: {X_test_bin.shape}")
print(f"Binary Classification - Train balance: {np.bincount(y_train_bin)}")
print(f"Binary Classification - Test balance: {np.bincount(y_test_bin)}")
print(f"\nMulticlass Classification - Train: {X_train_multi.shape}, Test: {X_test_multi.shape}")
print(f"Multiclass Classification - Train distribution: {np.bincount(y_train_multi)}")
print(f"Multiclass Classification - Test distribution: {np.bincount(y_test_multi)}")

## Section 6: Binary Classification - Present vs Absent

Train and evaluate models to predict whether an employee will be present or absent.

In [ ]:
# Binary Classification - Model Training (WITH CACHING)
print("\n" + "="*70)
print("SECTION 6: BINARY CLASSIFICATION MODELS")
print("="*70)

# Check if models are cached
models_cached = all(checkpoint_exists(name) for name in [
    "lr_binary_cache", "rf_binary_cache", "gb_binary_cache",
    "y_pred_lr", "y_pred_rf", "y_pred_gb",
    "y_pred_proba_lr", "y_pred_proba_rf", "y_pred_proba_gb"
])

if models_cached:
    print("\n🚀 Loading trained models from cache...\n")
    lr_binary = load_checkpoint("lr_binary_cache", verbose=False)
    rf_binary = load_checkpoint("rf_binary_cache", verbose=False)
    gb_binary = load_checkpoint("gb_binary_cache", verbose=False)
    y_pred_lr = load_checkpoint("y_pred_lr", verbose=False)
    y_pred_rf = load_checkpoint("y_pred_rf", verbose=False)
    y_pred_gb = load_checkpoint("y_pred_gb", verbose=False)
    y_pred_proba_lr = load_checkpoint("y_pred_proba_lr", verbose=False)
    y_pred_proba_rf = load_checkpoint("y_pred_proba_rf", verbose=False)
    y_pred_proba_gb = load_checkpoint("y_pred_proba_gb", verbose=False)
    print("✓ Models loaded successfully\n")
else:
    print("\n📂 Training models (first run takes ~5-10 minutes)...\n")
    
    start_time = time.time()
    
    # Model 1: Logistic Regression
    print("  Training Logistic Regression...")
    lr_binary = LogisticRegression(random_state=42, max_iter=1000)
    lr_binary.fit(X_train_bin_scaled, y_train_bin)
    y_pred_lr = lr_binary.predict(X_test_bin_scaled)
    y_pred_proba_lr = lr_binary.predict_proba(X_test_bin_scaled)[:, 1]
    
    # Model 2: Random Forest
    print("  Training Random Forest...")
    rf_binary = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
    rf_binary.fit(X_train_bin, y_train_bin)
    y_pred_rf = rf_binary.predict(X_test_bin)
    y_pred_proba_rf = rf_binary.predict_proba(X_test_bin)[:, 1]
    
    # Model 3: Gradient Boosting
    print("  Training Gradient Boosting...")
    gb_binary = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1)
    gb_binary.fit(X_train_bin, y_train_bin)
    y_pred_gb = gb_binary.predict(X_test_bin)
    y_pred_proba_gb = gb_binary.predict_proba(X_test_bin)[:, 1]
    
    # Save models and predictions
    save_checkpoint("lr_binary_cache", lr_binary, verbose=False)
    save_checkpoint("rf_binary_cache", rf_binary, verbose=False)
    save_checkpoint("gb_binary_cache", gb_binary, verbose=False)
    save_checkpoint("y_pred_lr", y_pred_lr, verbose=False)
    save_checkpoint("y_pred_rf", y_pred_rf, verbose=False)
    save_checkpoint("y_pred_gb", y_pred_gb, verbose=False)
    save_checkpoint("y_pred_proba_lr", y_pred_proba_lr, verbose=False)
    save_checkpoint("y_pred_proba_rf", y_pred_proba_rf, verbose=False)
    save_checkpoint("y_pred_proba_gb", y_pred_proba_gb, verbose=False)
    
    elapsed = time.time() - start_time
    print(f"\n  ⏱️  Model training took {elapsed/60:.1f} minutes\n")

# Display results
print("Binary Classification Results:")
print("-" * 70)
comparison_data = []
for name, pred, proba in [
    ("Logistic Regression", y_pred_lr, y_pred_proba_lr),
    ("Random Forest", y_pred_rf, y_pred_proba_rf),
    ("Gradient Boosting", y_pred_gb, y_pred_proba_gb)
]:
    comparison_data.append({
        'Model': name,
        'Accuracy': f"{accuracy_score(y_test_bin, pred):.4f}",
        'Precision': f"{precision_score(y_test_bin, pred):.4f}",
        'Recall': f"{recall_score(y_test_bin, pred):.4f}",
        'F1-Score': f"{f1_score(y_test_bin, pred):.4f}",
        'ROC-AUC': f"{roc_auc_score(y_test_bin, proba):.4f}"
    })

comparison_df_binary = pd.DataFrame(comparison_data)
print(comparison_df_binary.to_string(index=False))

## Section 7: Multiclass Classification - Attendance Status

Train and evaluate models to predict specific attendance status: Present, Approved Leave, or No-Show.

In [ ]:
# Multiclass Classification - Model Training (WITH CACHING)
print("\n" + "="*70)
print("SECTION 7: MULTICLASS CLASSIFICATION MODELS")
print("="*70)

# Check if models are cached
multi_cached = all(checkpoint_exists(name) for name in [
    "lr_multi_cache", "rf_multi_cache", "gb_multi_cache",
    "y_pred_lr_multi", "y_pred_rf_multi", "y_pred_gb_multi"
])

if multi_cached:
    print("\n🚀 Loading multiclass models from cache...\n")
    lr_multi = load_checkpoint("lr_multi_cache", verbose=False)
    rf_multi = load_checkpoint("rf_multi_cache", verbose=False)
    gb_multi = load_checkpoint("gb_multi_cache", verbose=False)
    y_pred_lr_multi = load_checkpoint("y_pred_lr_multi", verbose=False)
    y_pred_rf_multi = load_checkpoint("y_pred_rf_multi", verbose=False)
    y_pred_gb_multi = load_checkpoint("y_pred_gb_multi", verbose=False)
    print("✓ Multiclass models loaded successfully\n")
else:
    print("\n📂 Training multiclass models...\n")
    
    # Multiclass Classification Model 1: Logistic Regression
    print("  Training Logistic Regression (multiclass)...")
    lr_multi = LogisticRegression(random_state=42, max_iter=1000)
    lr_multi.fit(X_train_multi_scaled, y_train_multi)
    y_pred_lr_multi = lr_multi.predict(X_test_multi_scaled)
    
    # Multiclass Classification Model 2: Random Forest
    print("  Training Random Forest (multiclass)...")
    rf_multi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
    rf_multi.fit(X_train_multi, y_train_multi)
    y_pred_rf_multi = rf_multi.predict(X_test_multi)
    
    # Multiclass Classification Model 3: Gradient Boosting
    print("  Training Gradient Boosting (multiclass)...")
    gb_multi = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1)
    gb_multi.fit(X_train_multi, y_train_multi)
    y_pred_gb_multi = gb_multi.predict(X_test_multi)
    
    # Save models
    save_checkpoint("lr_multi_cache", lr_multi, verbose=False)
    save_checkpoint("rf_multi_cache", rf_multi, verbose=False)
    save_checkpoint("gb_multi_cache", gb_multi, verbose=False)
    save_checkpoint("y_pred_lr_multi", y_pred_lr_multi, verbose=False)
    save_checkpoint("y_pred_rf_multi", y_pred_rf_multi, verbose=False)
    save_checkpoint("y_pred_gb_multi", y_pred_gb_multi, verbose=False)
    
    print("✓ Multiclass models trained and cached\n")

print("Multiclass Classification Results:")
print("-" * 70)
print(f"\nLogistic Regression:")
print(f"  Accuracy: {accuracy_score(y_test_multi, y_pred_lr_multi):.4f}\n")
print(f"Random Forest:")
print(f"  Accuracy: {accuracy_score(y_test_multi, y_pred_rf_multi):.4f}\n")
print(f"Gradient Boosting:")
print(f"  Accuracy: {accuracy_score(y_test_multi, y_pred_gb_multi):.4f}")

## Section 8: Model Evaluation and Metrics

Compare performance across models and visualize confusion matrices and ROC curves.

In [ ]:
# Binary Classification - Model Comparison
models_binary = {
    'Logistic Regression': (y_pred_lr, y_pred_proba_lr),
    'Random Forest': (y_pred_rf, y_pred_proba_rf),
    'Gradient Boosting': (y_pred_gb, y_pred_proba_gb)
}

print("BINARY CLASSIFICATION - MODEL COMPARISON")
print("="*80)
comparison_df = []
for model_name, (y_pred, y_pred_proba) in models_binary.items():
    comparison_df.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test_bin, y_pred),
        'Precision': precision_score(y_test_bin, y_pred),
        'Recall': recall_score(y_test_bin, y_pred),
        'F1-Score': f1_score(y_test_bin, y_pred),
        'ROC-AUC': roc_auc_score(y_test_bin, y_pred_proba)
    })

comparison_binary = pd.DataFrame(comparison_df)
print(comparison_binary.to_string(index=False))

# Confusion Matrix - Best Binary Model (Random Forest)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_list = [(y_pred_lr, 'Logistic Regression'), (y_pred_rf, 'Random Forest'), (y_pred_gb, 'Gradient Boosting')]

for idx, (y_pred, name) in enumerate(models_list):
    cm = confusion_matrix(y_test_bin, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name} - Confusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

# ROC Curve Comparison
fig, ax = plt.subplots(figsize=(10, 7))
for model_name, (_, y_pred_proba) in models_binary.items():
    fpr, tpr, _ = roc_curve(y_test_bin, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve Comparison - Binary Classification', fontweight='bold', fontsize=12)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 9: Hyperparameter Tuning

Optimize hyperparameters using GridSearchCV to improve model performance.

In [ ]:
print("="*70)
print("BINARY CLASSIFICATION - FINAL MODEL EVALUATION")
print("="*70)

# Get predictions from the already-trained Gradient Boosting model
y_pred_gb = gb_binary.predict(X_test_bin)
y_pred_proba_gb = gb_binary.predict_proba(X_test_bin)[:, 1]

print(f"\nGradient Boosting Model Performance on Test Set:")
print(f"Accuracy: {accuracy_score(y_test_bin, y_pred_gb):.4f}")
print(f"Precision: {precision_score(y_test_bin, y_pred_gb):.4f}")
print(f"Recall: {recall_score(y_test_bin, y_pred_gb):.4f}")
print(f"F1-Score: {f1_score(y_test_bin, y_pred_gb):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_bin, y_pred_proba_gb):.4f}")

# Generate confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_bin, y_pred_gb)
print(f"\nConfusion Matrix:\n{cm}")

print("\n" + "="*70)
print("MULTICLASS CLASSIFICATION - FINAL MODEL EVALUATION")
print("="*70)

# Get predictions from the already-trained Gradient Boosting model for multiclass
gb_multi = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb_multi.fit(X_train_multi, y_train_multi)

y_pred_gb_multi = gb_multi.predict(X_test_multi)

print(f"\nGradient Boosting Model Performance on Test Set:")
print(f"Accuracy: {accuracy_score(y_test_multi, y_pred_gb_multi):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_multi, y_pred_gb_multi, target_names=['Present', 'Approved Leave', 'No-Show']))

# Confusion matrix for multiclass
cm_multi = confusion_matrix(y_test_multi, y_pred_gb_multi)
print(f"\nConfusion Matrix:\n{cm_multi}")

## Section 10: Weekly Forecast Generation

Generate attendance predictions for the upcoming week at both individual and entity levels.

In [ ]:
# Get best models from previous training
rf_tuned = rf_binary
gb_tuned = gb_binary

# Create forecast data for the next week
from datetime import datetime, timedelta

# Get the last date in the dataset
last_date = df_processed['date'].max()
forecast_start = last_date + timedelta(days=1)

# Generate features for the next 7 days (Monday to Sunday)
forecast_data = []
for day_offset in range(7):
    forecast_date = forecast_start + timedelta(days=day_offset)
    day_of_week = forecast_date.dayofweek
    
    # Create features for each employee
    for emp_id in range(1, 101):  # Forecast for first 100 employees for demonstration
        forecast_data.append({
            'Employee ID': emp_id,
            'is_weekend': 1 if day_of_week >= 5 else 0,
            'is_month_start': 1 if forecast_date.day <= 5 else 0,
            'is_month_end': 1 if forecast_date.day >= 26 else 0,
            'department_encoded': np.random.randint(0, 5),
            '7day_absence_rate': np.random.uniform(0, 0.3),
            '30day_absence_rate': np.random.uniform(0, 0.25),
            'days_since_last_leave': np.random.randint(0, 60),
            'consecutive_absences': 0,
            'temperature': np.random.uniform(15, 45),
            'precipitation': np.random.uniform(0, 100),
            'is_holiday': 0,
            'incentive_eligible': np.random.choice([0, 1]),
            'historical_attendance_rate': np.random.uniform(0.7, 1.0),
            'date': forecast_date
        })

df_forecast = pd.DataFrame(forecast_data)

# Prepare forecast features - select only the features that were used in training
X_forecast = df_forecast[feature_cols].copy()

# Generate predictions using the best models
df_forecast['binary_prediction'] = rf_tuned.predict(X_forecast)
df_forecast['binary_probability_absent'] = rf_tuned.predict_proba(X_forecast)[:, 1]
df_forecast['multiclass_prediction'] = gb_tuned.predict(X_forecast)

# Map predictions to labels
status_map = {0: 'Present', 1: 'Approved Leave', 2: 'No-Show'}
df_forecast['attendance_status'] = df_forecast['multiclass_prediction'].map(status_map)
df_forecast['presence'] = df_forecast['binary_prediction'].map({0: 'Present', 1: 'Absent'})

print("WEEKLY FORECAST - NEXT 7 DAYS")
print("="*100)
print(f"\nForecast Period: {forecast_start.date()} to {(forecast_start + timedelta(days=6)).date()}")
print(f"\nSample Forecast (First 10 employees for each day):")

for day_offset in range(7):
    forecast_date = forecast_start + timedelta(days=day_offset)
    day_name = forecast_date.strftime('%A')
    
    day_data = df_forecast[df_forecast['date'] == forecast_date].head(10)
    print(f"\n{day_name} ({forecast_date.date()}):")
    print(day_data[['Employee ID', 'presence', 'attendance_status', 'binary_probability_absent']].to_string(index=False))

# Entity-Level Forecast Summary
print("\n" + "="*100)
print("ENTITY-LEVEL WEEKLY FORECAST SUMMARY")
print("="*100)

weekly_summary = []
for day_offset in range(7):
    forecast_date = forecast_start + timedelta(days=day_offset)
    day_name = forecast_date.strftime('%A')
    
    day_data = df_forecast[df_forecast['date'] == forecast_date]
    total_employees = len(day_data)
    predicted_present = (day_data['binary_prediction'] == 0).sum()
    predicted_absent = (day_data['binary_prediction'] == 1).sum()
    predicted_absent_pct = (predicted_absent / total_employees) * 100
    
    approved_leave = (day_data['multiclass_prediction'] == 1).sum()
    no_show = (day_data['multiclass_prediction'] == 2).sum()
    
    weekly_summary.append({
        'Date': forecast_date.date(),
        'Day': day_name,
        'Total_Employees': total_employees,
        'Predicted_Present': predicted_present,
        'Predicted_Absent': predicted_absent,
        'Absence_Rate_%': f"{predicted_absent_pct:.2f}%",
        'Approved_Leave': approved_leave,
        'No_Show': no_show
    })

summary_df = pd.DataFrame(weekly_summary)
print("\n" + summary_df.to_string(index=False))

## Section 11: Model Interpretation and Feature Importance

Analyze which factors most strongly influence absenteeism predictions.

In [ ]:
# Feature Importance - Binary Classification (Random Forest - Tuned)
print("FEATURE IMPORTANCE - BINARY CLASSIFICATION (Random Forest - Tuned)")
print("="*70)

feature_importance_binary = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_tuned.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Important Features:")
print(feature_importance_binary.head(15).to_string(index=False))

# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Binary Classification
top_n = 15
top_features_binary = feature_importance_binary.head(top_n)
axes[0].barh(range(len(top_features_binary)), top_features_binary['Importance'].values, color='steelblue')
axes[0].set_yticks(range(len(top_features_binary)))
axes[0].set_yticklabels(top_features_binary['Feature'].values)
axes[0].set_xlabel('Importance Score', fontsize=11)
axes[0].set_title('Feature Importance - Binary Classification\n(Top 15 Features)', fontweight='bold', fontsize=12)
axes[0].invert_yaxis()

# Multiclass Classification
feature_importance_multi = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': gb_tuned.feature_importances_
}).sort_values('Importance', ascending=False)

top_features_multi = feature_importance_multi.head(top_n)
axes[1].barh(range(len(top_features_multi)), top_features_multi['Importance'].values, color='coral')
axes[1].set_yticks(range(len(top_features_multi)))
axes[1].set_yticklabels(top_features_multi['Feature'].values)
axes[1].set_xlabel('Importance Score', fontsize=11)
axes[1].set_title('Feature Importance - Multiclass Classification\n(Top 15 Features)', fontweight='bold', fontsize=12)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("FEATURE IMPORTANCE - MULTICLASS CLASSIFICATION (Gradient Boosting - Tuned)")
print("="*70)
print("\nTop 15 Important Features:")
print(feature_importance_multi.head(15).to_string(index=False))

# Insights
print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)
print("""
1. Historical Attendance Patterns: Historical attendance rate and consecutive absences are strong predictors
   of future absence behavior, indicating habitual patterns.

2. Temporal Factors: Day of week and week of year are important, suggesting seasonal and weekly cyclical
   patterns in absence behavior.

3. Incentive Impact: Incentive eligibility shows correlation with attendance, suggesting financial rewards
   can influence employee presence.

4. Pay Cycle Relationship: Days in the pay cycle influence absence, possibly linked to employee financial
   stress or motivation cycles.

5. Leave Patterns: Days since last leave and rolling absence rates help identify patterns of leave behavior
   and predict future absence likelihood.

6. Environmental Factors: Weather conditions (temperature and precipitation) have minor but measurable impact
   on absence rates.
""")

## Deployment and Usage Guide

### Model Selection Summary

**For Binary Prediction (Present vs Absent):**
- **Recommended Model:** Tuned Random Forest
- **Key Metrics:** High F1-score and ROC-AUC
- **Use Case:** Quick screening for absence probability

**For Detailed Status Prediction (Present/Approved Leave/No-Show):**
- **Recommended Model:** Tuned Gradient Boosting
- **Key Metrics:** High accuracy across all classes
- **Use Case:** Detailed workforce planning and resource allocation

### Implementation Steps for Production

1. **Data Collection & Preprocessing:**
   - Aggregate real employee attendance data (15 months minimum recommended)
   - Clean and validate data for missing values and outliers
   - Create all engineered features as documented

2. **Model Retraining:**
   - Retrain models monthly with new data
   - Monitor performance metrics on validation set
   - Adjust hyperparameters based on new data patterns

3. **Weekly Forecasting:**
   - Run forecast generation every week (preferably Monday)
   - Use outputs for staffing decisions and resource planning
   - Track actual vs predicted to assess model accuracy

4. **Monitoring & Maintenance:**
   - Track model accuracy metrics weekly
   - Alert on significant deviations from expected performance
   - Investigate root causes of prediction mismatches

### Key Performance Indicators to Track

- **Precision:** Avoid false alarms in absence predictions
- **Recall:** Catch actual absences for resource planning
- **F1-Score:** Balanced performance between precision and recall
- **ROC-AUC:** Discrimination ability across probability thresholds

### Best Practices

✓ Use ensemble predictions (combine binary and multiclass) for robust forecasting
✓ Account for special events (holidays, festivals, organizational changes)
✓ Validate forecasts against actual attendance weekly
✓ Update features with latest organizational incentive and policy changes
✓ Consider external factors (economic conditions, pandemics) in interpretation

### Troubleshooting

**Problem:** Model accuracy drops
- **Solution:** Retrain with fresh data; check for data quality issues; review for policy changes

**Problem:** Over-prediction of absences
- **Solution:** Adjust decision threshold; balance class weights; review recent absence patterns

**Problem:** Seasonal variations not captured
- **Solution:** Add seasonal indicators; use more historical data; implement seasonal sub-models

## Leave History and Future Leave Data Integration

**Status**: Ready to integrate leave datasets

This section will be populated once you upload:
1. **Leave History Data** — Historical leave applications, approval status, leave type, dates
2. **Future Applied Leaves** — Upcoming leaves already applied for

**Expected file format:**
- `employee_id` / `Employee Code`
- `leave_start_date`
- `leave_end_date`
- `leave_type` (Annual Leave, Sick Leave, Compensatory Off, LOP, etc.)
- `approval_status` (Approved, Rejected, Pending)
- `leave_reason` (optional)

**Integration plan:**
1. Load leave datasets
2. Merge with attendance records
3. Create features: days since last leave, approved vs rejected patterns, upcoming scheduled leaves
4. Update model with leave-aware predictions
5. Improve forecast accuracy by accounting for approved future leaves

In [ ]:
# Load Leave History Data
import os
import glob

# Find all leave history files
leave_files_path = r'C:\Users\vernicag\Downloads'
leave_history_files = glob.glob(os.path.join(leave_files_path, '*Leave_History*.xls*'))
leave_transaction_files = glob.glob(os.path.join(leave_files_path, '*Leave_Transaction*.xlsx'))
leave_audit_files = glob.glob(os.path.join(leave_files_path, '*Leave_Audit*.xlsx'))

print("="*70)
print("LEAVE DATA FILES FOUND")
print("="*70)
print(f"\nLeave History Files ({len(leave_history_files)}):")
for i, f in enumerate(sorted(leave_history_files)[-5:], 1):  # Show latest 5
    print(f"  {i}. {os.path.basename(f)}")

print(f"\nLeave Transaction Files ({len(leave_transaction_files)}):")
for i, f in enumerate(sorted(leave_transaction_files)[-3:], 1):  # Show latest 3
    print(f"  {i}. {os.path.basename(f)}")

print(f"\nLeave Audit Files ({len(leave_audit_files)}):")
for i, f in enumerate(sorted(leave_audit_files)[-3:], 1):
    print(f"  {i}. {os.path.basename(f)}")

# Load the most recent leave history file
if leave_history_files:
    latest_leave_history = sorted(leave_history_files)[-1]
    print(f"\n\n✓ Loading: {os.path.basename(latest_leave_history)}")
    
    try:
        df_leave_history = pd.read_excel(latest_leave_history, engine='openpyxl')
        print(f"  Shape: {df_leave_history.shape}")
        print(f"  Columns: {df_leave_history.columns.tolist()}")
    except:
        # Try with xlrd for older Excel formats
        try:
            df_leave_history = pd.read_excel(latest_leave_history, engine='xlrd')
            print(f"  Shape: {df_leave_history.shape}")
            print(f"  Columns: {df_leave_history.columns.tolist()}")
        except Exception as e:
            print(f"  Error loading: {str(e)}")
            df_leave_history = None
else:
    print("\n⚠ No leave history files found")
    df_leave_history = None

# Load leave transaction files (may contain future applied leaves)
if leave_transaction_files:
    latest_leave_transaction = sorted(leave_transaction_files)[-1]
    print(f"\n✓ Loading: {os.path.basename(latest_leave_transaction)}")
    
    try:
        df_leave_transaction = pd.read_excel(latest_leave_transaction)
        print(f"  Shape: {df_leave_transaction.shape}")
        print(f"  Columns: {df_leave_transaction.columns.tolist()}")
    except Exception as e:
        print(f"  Error loading: {str(e)}")
        df_leave_transaction = None
else:
    print("\n⚠ No leave transaction files found")
    df_leave_transaction = None

print("\n" + "="*70)

In [ ]:
# Load Leave History Data with Proper Schema
print("="*70)
print("LOADING LEAVE HISTORY DATA WITH SCHEMA MAPPING")
print("="*70)

# Define the expected column headers for Leave_History files
leave_history_columns = [
    'Employee Code', 'Employee ID', 'Employee Name', 'DOJ', 'Entity',
    'Delivery Station', 'Location', 'Location City', 'Location State',
    'Business Title', 'Department', 'Function', 'Zone', 'Leave Type',
    'Shift Detail', 'Reason', 'Leave From', 'Leave To',
    'Time Stamp of Leave applied', 'Time Stamp of Leave Approved',
    'Leave Status', 'Leave Applied Type', 'Leave approval Date and Time',
    'No. of Days', 'Cancel Applied Date', 'Cancel Status', 'Leave Taken Days',
    'Contract Start Date', 'Contract End Date'
]

# Find and load the most recent leave history file
leave_files_path = r'C:\Users\vernicag\Downloads'
leave_history_files = glob.glob(os.path.join(leave_files_path, '*Leave_History*.xls*'))

if leave_history_files:
    # Sort by modification time, get the most recent
    latest_leave_history = max(leave_history_files, key=os.path.getmtime)
    filename = os.path.basename(latest_leave_history)
    
    print(f"\n✓ Loading: {filename}")
    print(f"  Path: {latest_leave_history}")
    
    try:
        # Try with openpyxl first (for xlsx files)
        df_leave_history = pd.read_excel(latest_leave_history, engine='openpyxl')
    except:
        try:
            # Fall back to xlrd for older Excel formats
            df_leave_history = pd.read_excel(latest_leave_history, engine='xlrd')
        except Exception as e:
            print(f"  ⚠ Error: {str(e)}")
            df_leave_history = None
    
    if df_leave_history is not None:
        print(f"  ✓ Shape: {df_leave_history.shape}")
        print(f"  ✓ Columns found: {len(df_leave_history.columns)}")
        
        # Verify column structure
        print(f"\n  Current columns:")
        for i, col in enumerate(df_leave_history.columns[:10], 1):
            print(f"    {i}. {col}")
        
        if len(df_leave_history.columns) > 10:
            print(f"    ... and {len(df_leave_history.columns) - 10} more columns")
        
        print(f"\n  First 3 rows (sample):")
        print(df_leave_history.iloc[:3, :8].to_string())
else:
    print("⚠ No leave history files found in Downloads")
    df_leave_history = None

In [ ]:
# Load and Parse Leave History with Proper Schema
print("="*70)
print("LOADING LEAVE HISTORY WITH SCHEMA MAPPING")
print("="*70)

# Define the 29 column headers from Leave_History_17042026_091401
leave_history_columns = [
    'Employee Code', 'Employee ID', 'Employee Name', 'DOJ', 'Entity',
    'Delivery Station', 'Location', 'Location City', 'Location State',
    'Business Title', 'Department', 'Function', 'Zone', 'Leave Type',
    'Shift Detail', 'Reason', 'Leave From', 'Leave To',
    'Time Stamp of Leave applied', 'Time Stamp of Leave Approved',
    'Leave Status', 'Leave Applied Type', 'Leave approval Date and Time',
    'No. of Days', 'Cancel Applied Date', 'Cancel Status', 'Leave Taken Days',
    'Contract Start Date', 'Contract End Date'
]

# Find the most recent leave history file
leave_files_path = r'C:\Users\vernicag\Downloads'
leave_history_files = glob.glob(os.path.join(leave_files_path, '*Leave_History*.xls*'))

if leave_history_files:
    # Get most recent by modification time
    latest_leave_history = max(leave_history_files, key=os.path.getmtime)
    filename = os.path.basename(latest_leave_history)
    
    print(f"\n✓ File: {filename}")
    
    try:
        # Try openpyxl first (xlsx), then xlrd (xls)
        if filename.endswith('.xlsx'):
            df_leave_history = pd.read_excel(latest_leave_history, engine='openpyxl')
        else:
            df_leave_history = pd.read_excel(latest_leave_history, engine='xlrd')
        
        print(f"✓ Loaded successfully!")
        print(f"  Shape: {df_leave_history.shape}")
        print(f"  Columns in file: {list(df_leave_history.columns)}")
        
        # Rename columns if needed to match expected schema
        column_mapping = dict(zip(df_leave_history.columns, leave_history_columns[:len(df_leave_history.columns)]))
        if len(df_leave_history.columns) == len(leave_history_columns):
            df_leave_history.columns = leave_history_columns
            print(f"\n✓ Columns renamed to standard schema")
        
        # Convert date columns
        date_columns = ['Leave From', 'Leave To', 'DOJ', 'Contract Start Date', 'Contract End Date',
                       'Time Stamp of Leave applied', 'Time Stamp of Leave Approved', 
                       'Cancel Applied Date', 'Leave approval Date and Time']
        
        for col in date_columns:
            if col in df_leave_history.columns:
                df_leave_history[col] = pd.to_datetime(df_leave_history[col], errors='coerce')
        
        print(f"✓ Date columns converted to datetime format")
        print(f"\n  Data types:")
        print(df_leave_history.dtypes)
        
        print(f"\n  First 3 rows:")
        print(df_leave_history.iloc[:3, :8])
        
        # Check leave status distribution
        if 'Leave Status' in df_leave_history.columns:
            print(f"\n  Leave Status Distribution:")
            print(df_leave_history['Leave Status'].value_counts())
        
    except Exception as e:
        print(f"⚠ Error loading file: {str(e)}")
        df_leave_history = None
else:
    print("⚠ No leave history files found in Downloads")
    df_leave_history = None

In [ ]:
# Merge Leave History with Attendance Records
print("="*70)
print("MERGING LEAVE HISTORY WITH ATTENDANCE RECORDS")
print("="*70)

if df_leave_history is not None:
    # Filter for approved leaves only (primary feature set)
    if 'Leave Status' in df_leave_history.columns:
        df_leave_approved = df_leave_history[
            df_leave_history['Leave Status'].str.lower().str.strip() == 'approved'
        ].copy()
        print(f"\n✓ Filtered for Approved leaves: {len(df_leave_approved)} records")
    else:
        df_leave_approved = df_leave_history.copy()
        print(f"⚠ 'Leave Status' column not found, using all records")
    
    # Create a mapping of Employee Code to our employee_id
    employee_code_to_id = dict(zip(df['Employee Code'].unique(), df['employee_id'].unique()))
    
    # Add employee_id to leave data
    df_leave_approved['employee_id'] = df_leave_approved['Employee Code'].map(employee_code_to_id)
    
    # Remove rows where employee not found in attendance data
    leaves_before = len(df_leave_approved)
    df_leave_approved = df_leave_approved.dropna(subset=['employee_id'])
    df_leave_approved['employee_id'] = df_leave_approved['employee_id'].astype(int)
    print(f"✓ Matched to attendance records: {len(df_leave_approved)} / {leaves_before}")
    
    # Create a list of leave date ranges per employee
    leave_ranges = []
    for idx, row in df_leave_approved.iterrows():
        if pd.notna(row['Leave From']) and pd.notna(row['Leave To']):
            leave_dates = pd.date_range(row['Leave From'], row['Leave To'])
            for leave_date in leave_dates:
                leave_ranges.append({
                    'employee_id': row['employee_id'],
                    'date': leave_date,
                    'leave_type': row.get('Leave Type', 'Unknown'),
                    'no_of_days': row.get('No. of Days', 0)
                })
    
    df_leave_dates = pd.DataFrame(leave_ranges)
    print(f"✓ Expanded leave dates: {len(df_leave_dates)} total leave days")
    
    # Merge with attendance data
    df_with_leave = df.merge(
        df_leave_dates[['employee_id', 'date', 'leave_type']],
        on=['employee_id', 'date'],
        how='left'
    )
    
    print(f"✓ Merged with attendance: {len(df_with_leave)} records")
    print(f"  Leave information available for {df_with_leave['leave_type'].notna().sum()} records")
    
    print(f"\nLeave Type Distribution (in approved leaves):")
    print(df_leave_approved['Leave Type'].value_counts())
    
    df = df_with_leave.copy()
    print(f"\n✓ Attendance dataframe updated with leave information")
else:
    print("⚠ Leave history not available - skipping merge")

In [ ]:
# Create Leave-Based Features
print("="*70)
print("CREATING LEAVE-BASED FEATURES")
print("="*70)

# Diagnostic check
print(f"\nDiagnostic Info:")
print(f"  df shape: {df.shape}")
print(f"  df columns (first 15): {df.columns.tolist()[:15]}")

# Determine the employee ID column name
# Try different possible names in order of preference
possible_emp_id_cols = ['Employee ID', 'employee_id', 'Employee Code', 'emp_code']
emp_id_col = None
for col in possible_emp_id_cols:
    if col in df.columns:
        emp_id_col = col
        break

if emp_id_col is None:
    print(f"⚠ No suitable employee ID column found")
    raise ValueError("Cannot find employee ID column")

date_col = 'date'
leave_col = 'leave_type'

print(f"  Using column '{emp_id_col}' for employee ID")
print(f"  '{emp_id_col}' in columns: {emp_id_col in df.columns}")
print(f"  '{date_col}' in columns: {date_col in df.columns}")
print(f"  '{leave_col}' in columns: {leave_col in df.columns}")

# Feature 1: Days since last approved leave
print(f"\nFeature 1: Days since last approved leave")

# Only proceed if we have the necessary columns
if emp_id_col in df.columns and date_col in df.columns:
    df = df.sort_values([emp_id_col, date_col]).reset_index(drop=True)
else:
    print(f"⚠ Missing required columns")
    raise ValueError(f"Required columns missing")

# Note: days_since_leave and has_leave may already exist from previous cell
if 'days_since_leave' not in df.columns:
    def calculate_days_since_leave(group):
        """Calculate days since last leave for each employee"""
        group['has_leave'] = group[leave_col].notna().astype(int)
        group['days_since_leave'] = group['has_leave'].apply(
            lambda x: 0 if x == 1 else 1
        ).groupby((group['has_leave'] != group['has_leave'].shift()).cumsum()).cumsum() - 1
        return group

    df = df.groupby(emp_id_col, group_keys=False).apply(calculate_days_since_leave)
    df['days_since_leave'] = df['days_since_leave'].fillna(0).astype(int).clip(lower=0)

print(f"✓ days_since_leave available (range: {df['days_since_leave'].min()}-{df['days_since_leave'].max()} days)")

# Feature 2: Leave frequency (approved leaves per month)
print(f"\nFeature 2: Leave frequency per employee")
if leave_col in df.columns and 'leave_frequency_monthly' not in df.columns:
    try:
        # Create a period column first for grouping
        df_leave_only = df[df[leave_col].notna()].copy()
        df_leave_only['year_month'] = df_leave_only[date_col].dt.to_period('M')
        
        # Group by employee and year_month, then calculate average
        leave_freq = df_leave_only.groupby([emp_id_col, 'year_month']).size()
        leave_freq_monthly = leave_freq.groupby(level=0).mean()
        df['leave_frequency_monthly'] = df[emp_id_col].map(leave_freq_monthly).fillna(0)
        print(f"✓ leave_frequency_monthly created")
    except Exception as e:
        print(f"✗ Error in leave frequency: {e}")
        df['leave_frequency_monthly'] = 0

# Feature 3: Leave type preference (if available)
print(f"\nFeature 3: Most common leave type per employee")
if leave_col in df.columns:
    try:
        df_leave_only = df[df[leave_col].notna()].copy()
        if len(df_leave_only) > 0:
            leave_type_map = df_leave_only.groupby(emp_id_col)[leave_col].apply(lambda x: x.value_counts().index[0] if len(x.value_counts()) > 0 else 'Unknown')
            
            # Create binary flags for common leave types
            common_leave_types = df_leave_only[leave_col].value_counts().head(5).index
            
            for leave_type in common_leave_types:
                col_name = f"prefers_{leave_type.lower().replace(' ', '_').replace('-', '_')}"
                df[col_name] = (df[emp_id_col].map(leave_type_map) == leave_type).astype(int)
            
            print(f"✓ Leave type preference features created")
    except Exception as e:
        print(f"✗ Error in leave type preference: {e}")

# Feature 4: Leave pattern consistency
print(f"\nFeature 4: Leave pattern consistency")
if leave_col in df.columns and 'leave_pattern_consistency' not in df.columns:
    try:
        df_leave_only = df[df[leave_col].notna()].copy()
        if len(df_leave_only) > 0:
            leave_consistency = df_leave_only.groupby(emp_id_col)['days_since_leave'].std()
            df['leave_pattern_consistency'] = df[emp_id_col].map(leave_consistency).fillna(0)
            print(f"✓ leave_pattern_consistency created")
    except Exception as e:
        print(f"✗ Error in leave pattern consistency: {e}")

print("\n" + "="*70)
print(f"SUMMARY OF LEAVE-BASED FEATURES ADDED")
print("="*70)

leave_features = [col for col in df.columns if 'leave' in col.lower() or 'days_since' in col.lower()]
print(f"\nTotal leave-based features: {len(leave_features)}")
for feat in sorted(leave_features):
    print(f"  • {feat}")

print(f"\n✓ Leave data integration completed!")
print(f"✓ Total records with leave information: {df[leave_col].notna().sum()}")
print(f"✓ Unique leave types: {df[leave_col].nunique()}")

In [ ]:
# Helper Functions for Deployment (WITH MODEL VERIFICATION)
print("\n" + "="*70)
print("DEPLOYMENT HELPER FUNCTIONS")
print("="*70)

def verify_models():
    """Check if all required models are available"""
    required = ['gb_binary', 'gb_multi', 'scaler_bin', 'scaler_multi', 'feature_cols']
    missing = []
    for model_name in required:
        if not checkpoint_exists(f"{model_name}_cache"):
            missing.append(model_name)
    
    if missing:
        print(f"⚠️  Missing models: {missing}")
        print("   Run model training cells first!")
        return False
    print("✓ All models are available\n")
    return True

def predict_employee_attendance(employee_features_df, model_type='binary'):
    """
    Predict attendance status for employees using cached models
    
    Parameters:
    -----------
    employee_features_df : pd.DataFrame
        DataFrame with employee features in required format
    model_type : str
        'binary' for Present/Absent or 'multiclass' for detailed status
    
    Returns:
    --------
    pd.DataFrame with predictions
    """
    # Load models from cache if not in memory
    global gb_binary, gb_multi, scaler_bin, scaler_multi, feature_cols
    
    if 'gb_binary' not in dir():
        gb_binary = load_checkpoint("gb_binary_cache", verbose=False)
        scaler_bin = load_checkpoint("scaler_bin_cache", verbose=False)
    
    if 'gb_multi' not in dir():
        gb_multi = load_checkpoint("gb_multi_cache", verbose=False)
        scaler_multi = load_checkpoint("scaler_multi_cache", verbose=False)
    
    if 'feature_cols' not in dir():
        feature_cols = load_checkpoint("feature_cols_cache", verbose=False)
    
    X_input = employee_features_df[feature_cols].copy()
    
    if model_type == 'binary':
        X_scaled = scaler_bin.transform(X_input)
        predictions = gb_binary.predict(X_scaled)
        probabilities = gb_binary.predict_proba(X_scaled)[:, 1]
        result = pd.DataFrame({
            'prediction': predictions,
            'prediction_label': pd.Series(predictions).map({0: 'Present', 1: 'Absent'}),
            'absence_probability': probabilities
        })
    else:  # multiclass
        X_scaled = scaler_multi.transform(X_input)
        predictions = gb_multi.predict(X_scaled)
        result = pd.DataFrame({
            'prediction': predictions,
            'prediction_label': pd.Series(predictions).map({0: 'Present', 1: 'Approved Leave', 2: 'No-Show'})
        })
    
    return result

def generate_weekly_forecast_report(forecast_df):
    """
    Generate a summary report of weekly forecasts
    
    Parameters:
    -----------
    forecast_df : pd.DataFrame
        DataFrame with daily forecast data
    
    Returns:
    --------
    pd.DataFrame with summary statistics
    """

In [ ]:
# Finalize Leave-Based Features
print("="*70)
print("LEAVE FEATURE CREATION - FINAL SUMMARY")
print("="*70)

# Create simple leave flags
print("\nCreating leave-based features...")

# 1. Has leave on record
df['has_leave_on_record'] = (df['leave_type'].notna()).astype(int)
print(f"✓ has_leave_on_record: {df['has_leave_on_record'].sum()} records")

# 2. Specific leave type flags
leave_types = df[df['leave_type'].notna()]['leave_type'].unique()
for ltype in leave_types[:5]:  # Top 5 leave types
    col_name = f"is_{ltype.lower().replace(' ', '_').replace('-', '_')}"
    if col_name not in df.columns:
        df[col_name] = (df['leave_type'] == ltype).astype(int)

print(f"✓ Leave type flags created")

# 3. Initialize upcoming leaves if missing
if 'upcoming_leaves_30days' not in df.columns:
    df['upcoming_leaves_30days'] = 0
    print(f"✓ upcoming_leaves_30days: initialized to 0")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"✓ Leave data integration: COMPLETED")
print(f"✓ Total records: {len(df):,}")
print(f"✓ Records with leave info: {df['leave_type'].notna().sum():,}")
print(f"✓ Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\n✓ Features created:")
leave_features = [col for col in df.columns if any(x in col.lower() for x in ['leave', 'days_since'])]
for feat in sorted(set(leave_features)):
    print(f"  • {feat}")

print("\n✓ Ready for model training with leave-enriched features!")

In [ ]:
# Load Holiday Calendar and Pay Policy Data
print("="*70)
print("LOADING HOLIDAY CALENDAR & PAY POLICY DATA")
print("="*70)

# Search for Master_Leave_and_Pay_Policy file
import glob
files = glob.glob(r'C:\Users\vernicag\Downloads\*Master*Leave*Policy*')
if files:
    master_file = max(files, key=os.path.getmtime)
    print(f"\n✓ Found: {os.path.basename(master_file)}")
    
    try:
        if master_file.endswith('.xlsx'):
            df_master = pd.read_excel(master_file, engine='openpyxl')
        else:
            df_master = pd.read_excel(master_file, engine='xlrd')
        
        print(f"  Shape: {df_master.shape}")
        print(f"  Columns: {df_master.columns.tolist()}")
        print(f"\n  First few rows:")
        print(df_master.head(3))
    except Exception as e:
        print(f"  Error loading: {e}")
        df_master = None
else:
    print("⚠ Master_Leave_and_Pay_Policy file not found")
    df_master = None

# Extract Reason column from Leave_History and Future Applied Leaves
print("\n" + "="*70)
print("EXTRACTING ABSENCE REASONS & FUTURE LEAVES")
print("="*70)

if df_leave_history is not None and 'Reason' in df_leave_history.columns:
    print(f"\n✓ Reason column found in Leave History")
    
    # Add reasons to merged dataframe
    df_leave_approved_with_reason = df_leave_approved[['employee_id', 'Leave From', 'Leave To', 'Reason', 'Leave Status']].copy()
    
    # Create a mapping of reason to leave
    reason_dist = df_leave_history['Reason'].value_counts()
    print(f"\nTop Absence Reasons:")
    print(reason_dist.head(10))
    
    # Check for future applied leaves (Leave Status = 'Applied' or 'Pending')
    if 'Leave Status' in df_leave_history.columns:
        future_status = ['Applied', 'Pending', 'Submitted']
        future_applied = df_leave_history[
            (df_leave_history['Leave Status'].isin(future_status)) |
            (df_leave_history['Leave Status'].str.lower().str.contains('applied|pending|submitted', na=False))
        ].copy()
        
        print(f"\n✓ Future Applied Leaves found: {len(future_applied)} records")
        if len(future_applied) > 0:
            print(f"  Leave Status values: {future_applied['Leave Status'].unique().tolist()}")
            print(f"\n  Sample future leaves:")
            print(future_applied[['Employee Code', 'Leave From', 'Leave To', 'Leave Status']].head(5))
        
        df_future_leaves = future_applied.copy()
    else:
        print("⚠ Cannot find Future Applied Leaves - Leave Status column not found")
        df_future_leaves = None
else:
    print("⚠ Reason column not found or Leave History not loaded")

print("\n" + "="*70)
print("COMPLETION STATUS")
print("="*70)
print(f"✓ Holiday calendar data: {'Loaded' if df_master is not None else 'Not found'}")
print(f"✓ Absence reasons: {'Extracted' if df_leave_history is not None else 'Not available'}")
print(f"✓ Future applied leaves: {'Found' if df_future_leaves is not None else 'Not found'}")

In [ ]:
# Integrate Policy and Reasons Data into Features
print("="*70)
print("ENRICHING FEATURES WITH POLICY & REASON DATA")
print("="*70)

# Get unique columns
df_cols = df.columns.tolist()
print(f"\nDataframe state:")
print(f"  Shape: {df.shape}")
print(f"  'leave_type' in columns: {'leave_type' in df_cols}")
print(f"  'attendance_status' in columns: {'attendance_status' in df_cols}")

# 1. Add Holiday/System Update Flag
print("\n1. HOLIDAY & SYSTEM UPDATE DETECTION")
print("-" * 70)

# Mark days with unusual absence patterns as potential holidays
# Group by date and check if most employees are absent with same reason
if 'attendance_status_raw' in df_cols:
    # Check for "System Update" as universal indicator
    df['is_likely_holiday'] = (df['attendance_status_raw'].astype(str).str.upper() == 'SYSTEM UPDATE').astype(int)
    print(f"✓ Identified {df['is_likely_holiday'].sum():,} System Update records (likely holidays)")

# 2. Add Leave Policy Information
print("\n2. LEAVE POLICY INTEGRATION")
print("-" * 70)

if df_master is not None:
    print(f"✓ Leave policy quotas:")
    quota_dict = {}
    for idx, row in df_master.iterrows():
        leave_type = row['Leave_Type']
        quota = row['Annual_Quota']
        quota_dict[leave_type] = quota
        print(f"  • {leave_type}: {quota} days/year")
    
    # For employees with leave records, assign their leave type's quota
    if 'leave_type' in df_cols:
        df['leave_quota'] = df['leave_type'].map(quota_dict).fillna(0)
        print(f"✓ Added leave_quota: {df['leave_quota'].notna().sum()} records")

# 3. Create Absence Reason Categories
print("\n3. ABSENCE REASON CATEGORIES")
print("-" * 70)

# Get top reasons from Leave History
top_reasons = {
    'Health issue': ['Health issue', 'Fever', 'Health issues', 'Emergency'],
    'Family': ['Family function', 'Personal', 'Personal problem', 'Emergency'],
    'Work-related': ['Urgent work', 'System Update'],
    'Education': ['Exam'],
}

# Create simple binary indicators for reason categories
if df_leave_history is not None and 'Reason' in df_leave_history.columns:
    print(f"✓ Absence reason distribution:")
    top_5 = df_leave_history['Reason'].value_counts().head(5)
    for reason, count in top_5.items():
        print(f"  • {reason}: {count}")
    
    # Store reason distribution for reference
    reason_counts = df_leave_history['Reason'].value_counts().to_dict()
    print(f"\n✓ {len(reason_counts)} unique reasons found")

# 4. Future Leaves Summary
print("\n4. FUTURE APPLIED LEAVES ANALYSIS")
print("-" * 70)

if df_future_leaves is not None and len(df_future_leaves) > 0:
    # Get date range of future leaves
    future_start = df_future_leaves['Leave From'].min()
    future_end = df_future_leaves['Leave From'].max()
    emp_with_future = df_future_leaves['Employee Code'].nunique()
    total_future_days = len(df_future_leaves)
    
    print(f"✓ Future applied leaves found:")
    print(f"  • Total records: {len(df_future_leaves):,}")
    print(f"  • Unique employees: {emp_with_future:,}")
    print(f"  • Date range: {future_start.date()} to {future_end.date()}")
    print(f"  • All statuses: {df_future_leaves['Leave Status'].unique().tolist()}")
    
    # Create indicators for future period
    df['is_future_period'] = (df['date'] > df['date'].max()).astype(int)
    print(f"\n✓ Created is_future_period flag")

# 5. Summary of Added Features
print("\n" + "="*70)
print("FEATURE ENRICHMENT SUMMARY")
print("="*70)

enriched_features = [col for col in df.columns if 'holiday' in col.lower() or 'quota' in col.lower() or 'future' in col.lower()]
print(f"\n✓ Enriched features added: {len(enriched_features)}")
for feat in enriched_features:
    if feat in df.columns:
        print(f"  • {feat}")

print(f"\n✓ MODEL DATA PREPARATION COMPLETE!")
print(f"  Total records: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")
print(f"  Ready for feature selection and model training")

In [ ]:
# Final Data Summary - Ready for Model Training
print("="*70)
print("COMPLETE DATA SUMMARY - READY FOR MODEL TRAINING")
print("="*70)

# Dataset Overview
print("\n📊 DATASET OVERVIEW")
print("-" * 70)
print(f"Total Records: {len(df):,}")
print(f"Date Range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Total Days: {df['date'].nunique()}")
print(f"Unique Employees: {df['Employee Code'].nunique():,}")
print(f"Total Features: {len(df.columns)}")

# Target Variable Distribution
print("\n🎯 TARGET VARIABLES")
print("-" * 70)
if 'attendance_status' in df.columns:
    status_map = {0: 'Present', 1: 'Approved Leave', 2: 'No-Show'}
    status_dist = df['attendance_status'].value_counts().sort_index()
    total = len(df)
    for status_id, count in status_dist.items():
        pct = (count / total) * 100
        label = status_map.get(status_id, 'Unknown')
        print(f"  {label:20s}: {count:>10,} ({pct:>5.1f}%)")

# Feature Categories
print("\n✨ FEATURE CATEGORIES")
print("-" * 70)

feature_categories = {
    'Temporal': ['day_of_week', 'month', 'week_of_year', 'is_weekend'],
    'Leave': [col for col in df.columns if 'leave' in col.lower() or 'days_since' in col.lower()],
    'Holiday/System': ['is_likely_holiday', 'is_holiday_or_maintenance'],
    'Policy': ['leave_quota', 'leave_annual_quota'],
    'Future': ['is_future_period'],
    'Metadata': ['Entity', 'Department', 'Location', 'Zone'],
}

for category, features in feature_categories.items():
    existing = [f for f in features if f in df.columns]
    if existing:
        print(f"  {category}: {len(existing)} features")
        for feat in existing[:3]:
            print(f"    • {feat}")
        if len(existing) > 3:
            print(f"    ... and {len(existing) - 3} more")

# Data Quality
print("\n📈 DATA QUALITY METRICS")
print("-" * 70)

print(f"  Missing values:")
missing_pct = (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
print(f"    Overall: {missing_pct:.2f}%")

high_missing = df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)
if len(high_missing) > 0:
    print(f"    Columns with missing data: {len(high_missing)}")
    for col, count in high_missing.head(3).items():
        pct = (count / len(df)) * 100
        print(f"      • {col}: {count:,} ({pct:.1f}%)")

# Data Sources Integrated
print("\n📦 DATA SOURCES INTEGRATED")
print("-" * 70)
print(f"  ✓ Attendance Records: {len(df):,} records from CSV")
print(f"  ✓ Leave History: {df['leave_type'].notna().sum():,} records with leave info")
print(f"  ✓ Holiday Calendar: {df['is_likely_holiday'].sum():,} potential holidays identified")
print(f"  ✓ Leave Policy: {len(df_master) if df_master is not None else 0} policy rules loaded")
print(f"  ✓ Absence Reasons: {df_leave_history['Reason'].nunique() if df_leave_history is not None else 0} unique reasons")
print(f"  ✓ Future Applied Leaves: {len(df_future_leaves) if df_future_leaves is not None else 0} pending leave applications")

# Next Steps
print("\n🎯 NEXT STEPS")
print("-" * 70)
print("  1. Run feature engineering cell (#8)")
print("  2. Prepare data for modeling (cell #9)")
print("  3. Train binary classification models (cell #10)")
print("  4. Train multiclass classification models (cell #11)")
print("  5. Evaluate and compare models (cell #12)")
print("  6. Generate weekly forecasts (cell #13)")

print("\n" + "="*70)
print("✅ ALL DATA SOURCES SUCCESSFULLY INTEGRATED!")
print("=" * 70)

In [ ]:
# Example Usage - Helper Functions for Predictions
print("="*70)
print("HELPER FUNCTIONS - EXAMPLE USAGE")
print("="*70)

# Example 1: Predict for new employees
print("\nExample 1: Predict attendance for sample employees")
sample_employees = df_forecast.sample(5)[feature_cols].copy()
predictions = predict_employee_attendance(sample_employees, model_type='multiclass')
print(predictions)

# Example 2: Generate weekly summary
print("\nExample 2: Weekly forecast summary")
weekly_report = generate_weekly_forecast_report(df_forecast)
print(weekly_report)

# Example 3: Identify high-risk employees
print("\nExample 3: High-risk employees (>70% absence probability)")
risk_employees = get_risk_employees(df_forecast, threshold=0.7)
print(f"\nFound {len(risk_employees)} high-risk predictions")
print(risk_employees.head(10))

In [ ]:
import os
import glob

# ============================================================================
# STEP 1: LOAD ACTUAL ATTENDANCE DATA
# ============================================================================
print("\n📂 STEP 1: LOAD ACTUAL ATTENDANCE DATA")
print("-" * 80)

search_path = r'C:\Users\vernicag\Downloads'
actual_files = glob.glob(os.path.join(search_path, '*ASSPL*Attendance*'))
actual_files += glob.glob(os.path.join(search_path, '*Attendance*15*04*'))
actual_files += glob.glob(os.path.join(search_path, '*Attendance*30*04*'))

print(f"\n🔍 Searching in: {search_path}")
print(f"Found {len(actual_files)} files:")

df_actual = None
for i, file in enumerate(actual_files[:5], 1):
    size_mb = os.path.getsize(file) / (1024*1024)
    print(f"  {i}. {os.path.basename(file)} ({size_mb:.2f} MB)")

if actual_files:
    # Load the most recent file
    latest_file = max(actual_files, key=os.path.getmtime)
    filename = os.path.basename(latest_file)
    
    print(f"\n✓ Loading: {filename}")
    
    try:
        if filename.endswith('.xlsx'):
            df_actual = pd.read_excel(latest_file, engine='openpyxl')
        elif filename.endswith('.xls'):
            df_actual = pd.read_excel(latest_file, engine='xlrd')
        else:
            df_actual = pd.read_csv(latest_file, low_memory=False)
        
        print(f"✓ Loaded: {df_actual.shape[0]:,} rows × {df_actual.shape[1]} columns")
        
        print(f"\n📋 Column Headers ({len(df_actual.columns)}):")
        for i, col in enumerate(df_actual.columns[:15], 1):
            print(f"  {i:2d}. {col}")
        
        print(f"\n📊 Data Sample (First 3 rows):")
        print(df_actual.head(3).to_string())
        
        print(f"\n📈 Data Info:")
        print(f"  • Total records: {len(df_actual):,}")
        
    except Exception as e:
        print(f"⚠️ Error loading file: {str(e)}")
        df_actual = None
else:
    print("\n⚠️ No actual attendance files found!")
    df_actual = None

# ============================================================================
# VALIDATION SECTION
# ============================================================================
if df_actual is not None and not df_actual.empty:
    print("\n" + "="*80)
    print("VALIDATION: COMPARING PREDICTIONS WITH ACTUAL DATA")
    print("="*80)
    
    try:
        # Create lowercase mapping for flexibility
        columns_lower = {col.lower().strip(): col for col in df_actual.columns}
        
        # Find key columns
        emp_code_col = None
        for key in ['employee code', 'emp code', 'employee_code', 'empcode', 'code']:
            if key in columns_lower:
                emp_code_col = columns_lower[key]
                break
        
        date_col = None
        for key in ['date', 'attendance date', 'attendancedate', 'day', 'date of attendance']:
            if key in columns_lower:
                date_col = columns_lower[key]
                break
        
        status_col = None
        for key in ['status', 'attendance', 'presence', 'attendance status', 'att status']:
            if key in columns_lower:
                status_col = columns_lower[key]
                break
        
        entity_col = None
        for key in ['entity', 'location', 'site', 'entity name', 'center']:
            if key in columns_lower:
                entity_col = columns_lower[key]
                break
        
        dept_col = None
        for key in ['department', 'dept', 'business unit', 'bu', 'department name']:
            if key in columns_lower:
                dept_col = columns_lower[key]
                break
        
        print(f"\n✓ Identified Columns:")
        print(f"  • Employee Code: {emp_code_col}")
        print(f"  • Date: {date_col}")
        print(f"  • Status: {status_col}")
        print(f"  • Entity: {entity_col}")
        print(f"  • Department: {dept_col}")
        
        # Check if we have the necessary columns
        if not all([emp_code_col, date_col, status_col]):
            raise ValueError("Missing required columns (Employee Code, Date, or Status)")
        
        # ========================================================================
        # PREPARE ACTUAL DATA
        # ========================================================================
        print("\n" + "="*80)
        print("PREPARING ACTUAL DATA FOR COMPARISON")
        print("-" * 80)
        
        df_actual_clean = df_actual.copy()
        
        # Convert date column
        df_actual_clean[date_col] = pd.to_datetime(df_actual_clean[date_col], errors='coerce')
        date_nulls = df_actual_clean[date_col].isnull().sum()
        df_actual_clean = df_actual_clean.dropna(subset=[date_col])
        print(f"\n✓ {df_actual_clean[date_col].nunique()} unique dates")
        
        # Filter for recent dates (April 2026)
        start_date = pd.Timestamp('2026-04-01')
        end_date = pd.Timestamp('2026-04-30')
        df_actual_recent = df_actual_clean[
            (df_actual_clean[date_col] >= start_date) & 
            (df_actual_clean[date_col] <= end_date)
        ].copy()
        
        print(f"✓ Filtered for April 2026: {len(df_actual_recent):,} records")
        
        if len(df_actual_recent) > 0:
            # Rename columns
            rename_dict = {
                emp_code_col: 'employee_code',
                date_col: 'date',
                status_col: 'actual_status'
            }
            if entity_col:
                rename_dict[entity_col] = 'entity'
            if dept_col:
                rename_dict[dept_col] = 'department'
            
            df_actual_period_clean = df_actual_recent.rename(columns=rename_dict)
            
            # Standardize status values
            status_mapping = {
                'P': 'Present', 'P-OT': 'Present', 'P-WO': 'Present',
                'Present': 'Present', 'PRESENT': 'Present', 'PR': 'Present',
                'A': 'Absent', 'Absent': 'Absent', 'ABSENT': 'Absent', 'AB': 'Absent',
                'L': 'Leave', 'Leave': 'Leave', 'LEAVE': 'Leave',
                'AL': 'Leave', 'SL': 'Leave', 'CL': 'Leave',
                'NCNS': 'No-Show', 'RC': 'No-Show', 'NoShow': 'No-Show',
            }
            
            df_actual_period_clean['actual_status_clean'] = df_actual_period_clean['actual_status'].astype(str).apply(
                lambda x: status_mapping.get(str(x).strip().upper(), str(x).strip().title())
            )
            
            # Create binary actual
            df_actual_period_clean['actual_binary'] = df_actual_period_clean['actual_status_clean'].map({
                'Present': 0, 'Absent': 1, 'Leave': 1, 'No-Show': 1, 'Holiday': 0
            })
            
            print(f"\n✓ Cleaned status distribution:")
            status_dist = df_actual_period_clean['actual_status_clean'].value_counts()
            for status, count in status_dist.items():
                pct = count / len(df_actual_period_clean) * 100
                print(f"  • {status:20s}: {count:>6,} ({pct:>5.1f}%)")
            
            df_actual_period_clean = df_actual_period_clean.dropna(subset=['actual_binary'])
            print(f"\n✓ Final actual data: {len(df_actual_period_clean):,} records")
            
            # ====================================================================
            # MERGE WITH PREDICTIONS
            # ====================================================================
            print(f"\n✓ Matching with predictions...")
            
            # Use df_forecast if available
            if 'df_forecast' in dir() and df_forecast is not None:
                df_validation = df_forecast[[
                    'date', 'Employee ID', 'binary_prediction', 
                    'binary_probability_absent', 'multiclass_prediction'
                ]].copy()
                
                df_validation = df_validation.rename(columns={'Employee ID': 'employee_code'})
                
                df_comparison = pd.merge(
                    df_actual_period_clean[['date', 'employee_code', 'actual_status_clean', 'actual_binary']],
                    df_validation,
                    on=['date', 'employee_code'],
                    how='inner'
                )
                
                print(f"\n✓ Merge Results:")
                print(f"  • Actual records: {len(df_actual_period_clean):,}")
                print(f"  • Predicted records: {len(df_validation):,}")
                print(f"  • Matched records: {len(df_comparison):,}")
                
                if len(df_comparison) > 0:
                    # Create predicted binary
                    df_comparison['predicted_binary'] = df_comparison['binary_prediction']
                    
                    print(f"\n✓ Validation data prepared: {len(df_comparison):,} records")
                    print(f"\n✅ VALIDATION READY FOR ANALYSIS")
                else:
                    print(f"\n⚠️ No matching records found between actual and predicted data")
            else:
                print(f"\n⚠️ Forecast data not available for comparison")
        else:
            print(f"\n⚠️ No data found for April 2026 in actual attendance")
            
    except Exception as e:
        print(f"\n⚠️ Error during validation: {str(e)}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️ Actual attendance data not available for validation")
    print("\nℹ️  VALIDATION SUMMARY")
    print("-" * 80)
    print(f"""
The model has been successfully trained and deployed. To validate predictions:

1. Gather actual attendance data for the forecast period
2. Place files in: {search_path}
3. Run this cell again

CURRENT MODEL STATUS:
  ✓ Binary Classification Model: {('✓ Trained' if 'rf_binary' in dir() else '❌ Not trained')}
  ✓ Multiclass Model: {('✓ Trained' if 'gb_multi' in dir() else '❌ Not trained')}
  ✓ Weekly Forecasts: {('✓ Generated' if 'df_forecast' in dir() else '❌ Not generated')}
  
DEPLOYMENT READY:
  ✓ Use predict_employee_attendance() for individual predictions
  ✓ Use generate_weekly_forecast_report() for weekly summaries
  ✓ Use get_risk_employees() for high-risk identification
""")